# HidroData SIN v1.0 — 01 · Ingesta, limpieza y calidad

**Entrega final · Criterios 5 y 6 de la rúbrica.** Este notebook es la ventana al pipeline: lo ejecuta (o reutiliza su salida) y muestra la evidencia de cada etapa. La lógica vive en `src/` para que se ejecute igual cada vez; aquí solo se lee y se explica.

| Etapa | Script | Transformaciones del Plan (§8.2) | Salida |
|---|---|---|---|
| Ingesta | `src/ingesta.py` | Une los 5 archivos anuales por colección | `data/processed/ingesta/*.parquet` |
| Normalización y limpieza | `src/normalizacion.py` + `src/limpieza.py` | T1 formato numérico, T2 mes → fecha, T3 nombres, validez e imputación con bitácora | `data/processed/normalizado/*.parquet` |
| Calidad | `src/calidad.py` | Completitud, unicidad, validez y cobertura por colección | `data/processed/calidad/*.csv` |

**Requisito:** el `.rar` de `data/raw/` descomprimido (ver README). Si `data/processed/` ya existe, no se vuelve a correr el pipeline (≈ 5 min); para forzarlo, poner `FORZAR = True`.

In [1]:
from pathlib import Path
import subprocess, sys
import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_pipeline.py").exists())
PROC = RAIZ / "data" / "processed"
FORZAR = False

if FORZAR or not (PROC / "calidad" / "metricas_calidad.csv").exists():
    r = subprocess.run([sys.executable, "run_pipeline.py"], cwd=RAIZ, capture_output=True, text=True)
    print(r.stdout[-1500:]); assert r.returncode == 0, r.stderr[-2000:]
else:
    print("Se reutiliza la salida existente de run_pipeline.py")

Se reutiliza la salida existente de run_pipeline.py


## 1. Ingesta: filas por colección y archivo de origen
Cada colección viene en 5 archivos anuales (2022–2026). La ingesta los une sin cambiar valores y conserva `archivo_origen` para el linaje.

In [2]:
filas = []
for f in sorted((PROC / "ingesta").glob("*.parquet")):
    d = pd.read_parquet(f, columns=["archivo_origen"])
    filas.append({"coleccion": f.stem, "filas": len(d), "archivos": d["archivo_origen"].nunique()})
ingesta = pd.DataFrame(filas).set_index("coleccion")
ingesta.loc["TOTAL"] = ingesta.sum()
ingesta

,filas,archivos
coleccion,,
Aportes,2268,5
Capacidad,1383086,5
Demanda,1730,5
Generacion,505643,5
Precio,199213,5
Reservas,1731,5
Vertimentos,3989,5
TOTAL,2097660,35


## 2. Normalización: esquema estándar
Tras la normalización todas las colecciones tienen `fecha` como fecha real, medidas numéricas con la unidad en el nombre y textos sin espacios sobrantes.

In [3]:
esquema = []
for f in sorted((PROC / "normalizado").glob("*.parquet")):
    d = pd.read_parquet(f)
    esquema.append({"dataset": f.stem, "filas": len(d), "columnas": d.shape[1],
                    "desde": d["fecha"].min().date(), "hasta": d["fecha"].max().date(),
                    "medidas": ", ".join(c for c in d.select_dtypes("number").columns if c != "anio_archivo")[:110]})
pd.DataFrame(esquema).set_index("dataset")

,filas,columnas,desde,hasta,medidas
dataset,,,,,
aportes,2268,14,2022-01-01,2026-06-01,"aportes_energia_kwh, aportes_caudal_m3_s, apor..."
capacidad,1383086,11,2022-01-01,2026-09-26,capacidad_efectiva_neta_kw
compras_bolsa,199213,29,2022-01-01,2026-09-23,"hora_00, hora_01, hora_02, hora_03, hora_04, h..."
demanda,1730,8,2022-01-01,2026-09-26,"demanda_energia_sin_kwh, generacion_kwh, deman..."
generacion,505643,36,2022-01-01,2026-09-25,"hora_00, hora_01, hora_02, hora_03, hora_04, h..."
reservas,1731,4,2022-01-01,2026-09-27,volumen_util_diario_pct
vertimientos,3989,7,2022-01-01,2026-09-27,"vertimientos_volumen_miles_m3, vertimientos_en..."


## 3. Limpieza: qué se invalidó y qué se imputó
Ningún valor se cambia en silencio. Cada valor inválido queda en `incidentes_validez.csv` con su valor original, y cada imputación en `imputaciones.csv` con su método. El Data Steward revisa estas bitácoras (Plan §4.5, mejora M3 del BPMN).

In [4]:
incidentes = pd.read_csv(PROC / "calidad" / "incidentes_validez.csv")
print(f"Incidentes de validez: {len(incidentes)}")
incidentes

Incidentes de validez: 1


,coleccion,fecha,entidad,medida,valor_original,regla
0,aportes,2026-06-01,QUEBRADONA,aportes_energia_kwh,-676298.56,>= 0


In [5]:
imput = pd.read_csv(PROC / "calidad" / "imputaciones.csv")
resumen_imput = imput.groupby(["coleccion", "metodo"])["n_celdas"].sum().rename("celdas").reset_index()
resumen_imput["regla"] = resumen_imput["metodo"].map({
    "proporcion_caudal": "Energía negativa invalidada → caudal × mediana energía/caudal del río",
    "cero_sin_flujo": "Importación/exportación vacía = sin flujo (0)",
    "dia_anterior": "Generación SIN vacía → valor del día anterior",
    "interpolacion_horaria": "Una hora vacía en la fila → promedio de sus vecinas",
    "cero_sin_compra": "Hora vacía en compras 2026 = sin compra (0), como en 2022–2025"})
resumen_imput

,coleccion,metodo,celdas,regla
0,aportes,proporcion_caudal,1,Energía negativa invalidada → caudal × mediana...
1,compras_bolsa,cero_sin_compra,277444,"Hora vacía en compras 2026 = sin compra (0), c..."
2,demanda,cero_sin_flujo,499,Importación/exportación vacía = sin flujo (0)
3,demanda,dia_anterior,1,Generación SIN vacía → valor del día anterior
4,generacion,interpolacion_horaria,59,Una hora vacía en la fila → promedio de sus ve...


## 4. Calidad: métricas por colección
`calidad.py` mide sobre la capa normalizada. Las reglas formales por ID (A-01, R-01, GN-03…) se miden sobre la capa cruda en `documentacion/PlanDeGobierno/` (Excel S2 y `calidad_tablas_grandes.py`).

In [6]:
m = pd.read_csv(PROC / "calidad" / "metricas_calidad.csv")
print("Ejecución:", m["fecha_ejecucion"].iloc[0])
claves = ["registros", "duplicados_llave", "registros_medida_negativa", "registros_pct_fuera_0_100",
          "registros_fecha_fuera_rango", "valores_invalidados_normalizacion", "valores_imputados_normalizacion",
          "cobertura_fechas_pct", "registros_validos_pct"]
tabla = m[m["regla"].isin(claves)].pivot(index="coleccion", columns="regla", values="valor")[[c for c in claves]]
tabla

Ejecución: 2026-11-05 18:01:52


regla,registros,duplicados_llave,registros_medida_negativa,registros_pct_fuera_0_100,registros_fecha_fuera_rango,valores_invalidados_normalizacion,valores_imputados_normalizacion,cobertura_fechas_pct,registros_validos_pct
coleccion,,,,,,,,,
GLOBAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0
aportes,2268,0,0,0,0,1,1,100.0,100.0
capacidad,1383086,0,0,0,0,0,0,100.0,100.0
compras_bolsa,199213,0,0,0,0,0,277444,100.0,100.0
demanda,1730,0,0,0,0,0,500,100.0,100.0
generacion,505643,0,0,0,0,0,59,100.0,100.0
reservas,1731,0,0,0,0,0,0,100.0,100.0
vertimientos,3989,0,0,0,0,0,0,no aplica: serie esporádica,100.0


In [7]:
glob = m[m["coleccion"] == "GLOBAL"].set_index("regla")["valor"]
print(glob.to_string())

regla
registros_invalidos      0 de 2097660
registros_validos_pct           100.0


## 5. Lectura

- Las 7 colecciones de XM (≈ 2,1 millones de registros) pasan las reglas de validez de la capa normalizada; la meta O3 del Plan (≥ 98 % de registros válidos) se cumple.
- Solo hubo **1 valor inválido** en toda la fuente (energía negativa de QUEBRADONA, jun-2026), invalidado y reconstruido con bitácora.
- Las imputaciones son pocas y cada una tiene regla y método; las de compras en bolsa son de formato (2026 publica vacío lo que antes era 0).
- Siguiente paso del linaje: `04_eda.ipynb` lee `data/processed/normalizado/` y produce los indicadores del dashboard y la tabla de alerta de Power Automate.